![dvd_image](dvd_image.jpg)

A DVD rental company needs your help! They want to figure out how many days a customer will rent a DVD for based on some features and has approached you for help. They want you to try out some regression models which will help predict the number of days a customer will rent a DVD for. The company wants a model which yeilds a MSE of 3 or less on a test set. The model you make will help the company become more efficient inventory planning.

The data they provided is in the csv file `rental_info.csv`. It has the following features:
- `"rental_date"`: The date (and time) the customer rents the DVD.
- `"return_date"`: The date (and time) the customer returns the DVD.
- `"amount"`: The amount paid by the customer for renting the DVD.
- `"amount_2"`: The square of `"amount"`.
- `"rental_rate"`: The rate at which the DVD is rented for.
- `"rental_rate_2"`: The square of `"rental_rate"`.
- `"release_year"`: The year the movie being rented was released.
- `"length"`: Lenght of the movie being rented, in minuites.
- `"length_2"`: The square of `"length"`.
- `"replacement_cost"`: The amount it will cost the company to replace the DVD.
- `"special_features"`: Any special features, for example trailers/deleted scenes that the DVD also has.
- `"NC-17"`, `"PG"`, `"PG-13"`, `"R"`: These columns are dummy variables of the rating of the movie. It takes the value 1 if the move is rated as the column name and 0 otherwise. For your convinience, the reference dummy has already been dropped.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Import any additional modules and start coding below
df = pd.read_csv('rental_info.csv')
df["rental_length_days"] = (pd.to_datetime(df['return_date']) - pd.to_datetime(df['rental_date'])).dt.days
df['deleted_scenes'] = np.where(df['special_features'].str.contains("Deleted Scenes"), 1, 0 )
df['behind_the_scenes'] = np.where(df['special_features'].str.contains("Behind the Scenes"), 1, 0 )

X = df.drop(columns=['rental_length_days', 'rental_date', 'return_date','special_features'])
y = df['rental_length_days']

print(X.info())
print(y.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15861 entries, 0 to 15860
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   amount             15861 non-null  float64
 1   release_year       15861 non-null  float64
 2   rental_rate        15861 non-null  float64
 3   length             15861 non-null  float64
 4   replacement_cost   15861 non-null  float64
 5   NC-17              15861 non-null  int64  
 6   PG                 15861 non-null  int64  
 7   PG-13              15861 non-null  int64  
 8   R                  15861 non-null  int64  
 9   amount_2           15861 non-null  float64
 10  length_2           15861 non-null  float64
 11  rental_rate_2      15861 non-null  float64
 12  Deleted Scenes     15861 non-null  int64  
 13  Behind the Scenes  15861 non-null  int64  
dtypes: float64(8), int64(6)
memory usage: 1.7 MB
None
<class 'pandas.core.series.Series'>
RangeIndex: 15861 entries, 0 to 

In [6]:
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error

mse = {}

In [8]:
linear_reg =  LinearRegression()
linear_reg.fit(X_train_scaled, y_train)
y_pred = linear_reg.predict(X_test_scaled)
mse_linear_reg = mean_squared_error(y_test, y_pred)

mse['linear_reg'] = (mse_linear_reg,linear_reg)

In [9]:
param_rf_dist = {'n_estimators': np.arange(1,101,1),
          'max_depth':np.arange(1,11,1)}

rand_search = RandomizedSearchCV(RandomForestRegressor(),
                                 param_distributions=param_rf_dist,
                                 cv=5,
                                 random_state=42)
rand_search.fit(X_train_scaled, y_train)
hyper_params = rand_search.best_params_
print(rand_search.best_params_)
best_rf = rand_search.best_estimator_
best_rf.fit(X_train_scaled, y_train)
y_pred = best_rf.predict(X_test_scaled)
mse_random_forest = mean_squared_error(y_test,y_pred)
print(mse_random_forest)

mse['random_forest'] = (mse_random_forest,best_rf)

{'n_estimators': 61, 'max_depth': 9}
2.2209243905355125


In [10]:
params_dt_dist = {
    'max_depth' :np.arange(1,11),
    'min_samples_leaf' : np.arange(1,11),
    'min_samples_split': np.arange(2,11),
    'max_features': [None, 'sqrt', 'log2'],

}

rand_search = RandomizedSearchCV(DecisionTreeRegressor(),
                                 param_distributions=params_dt_dist,
                                 cv=5,
                                 random_state=42)
rand_search.fit(X_train_scaled, y_train)
hyper_params = rand_search.best_params_
print(rand_search.best_params_)
best_dt = rand_search.best_estimator_
best_dt.fit(X_train_scaled, y_train)
y_pred = best_dt.predict(X_test_scaled)
mse_decision_tree = mean_squared_error(y_test,y_pred)
print(mse_decision_tree)

mse['decision_tree'] = (mse_decision_tree,best_dt)

{'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 9}
2.380571909118065


In [11]:
params_knn_dist = {
    'n_neighbors': np.arange(1, 11),
    'weights' : ['uniform', 'distance'],
    'p' : np.arange(1, 11),
}

rand_search = RandomizedSearchCV(KNeighborsRegressor(),
                                 param_distributions=params_knn_dist,
                                 cv=5,
                                 random_state=42)
rand_search.fit(X_train_scaled, y_train)
hyper_params = rand_search.best_params_
print(rand_search.best_params_)
best_knn = rand_search.best_estimator_
best_knn.fit(X_train_scaled, y_train)
y_pred = best_knn.predict(X_test_scaled)
mse_knn = mean_squared_error(y_test,y_pred)
print(mse_knn)

mse['knn'] = (mse_knn,best_knn)

{'weights': 'distance', 'p': 8, 'n_neighbors': 6}
2.1885567571107125


## Find the best fitted model

Comparing based on the MSE for each model

$$\frac{1}{n}\sum_{i=1}^{n}(y_{true}(i) - y_{pred}(i))^{2}$$

To get both the key and the value together as a tuple, use min() on the dictionary's items() and provide a lambda function to compare the values (index 1 of the tuple).

In [16]:
best_name, (best_mse, best_model) = min(mse.items(), key=lambda x: x[1][0])
print(best_name, best_mse)
print(best_model)

('knn', (2.1885567571107125, KNeighborsRegressor(n_neighbors=6, p=8, weights='distance')))
KNeighborsRegressor(n_neighbors=6, p=8, weights='distance')
